# Pagila SQL and Python Exercises

The following exercises are generated using ChatGPT based on the Pagila database to practise SQL and Python programming for data science jobs in customer relationship management (CRM) and financial crime detection risk analytics.

In [12]:
import numpy as np
import pandas as pd
from sqlalchemy import URL, create_engine

In [ ]:
class DB:
    
    def __init__(self, drivername, database, host, port, username, password):
        self.url = URL.create(
            drivername = drivername,
            database = database,
            host = host,
            port = port,
            username = username,
            password = password
        )
        self.engine = create_engine(self.url)

    def execute_sql(self, sql):
        return pd.read_sql(sql, self.engine)

    def close_connection(self):
        self.engine.dispose()

In [59]:
pagila = DB(
    drivername = "postgresql+psycopg2",
    database = "pagila",
    host = "localhost",
    port = "5432",
    username = "postgres",
    password = "LZZcode2026!"
)

ex1_data = pagila.execute_sql(
    """ 
    SELECT
        c.store_id,
        c.customer_id,
        p.amount
    FROM customer AS c
    JOIN payment AS p
        ON c.customer_id = p.customer_id
    """
)

ex2_data = pagila.execute_sql(
    """ 
    SELECT
        customer_id,
        rental_date,
        rental_id
    FROM rental
    """
)

pagila.close_connection()

## Question 1

🧪 Question 1 — High-Value Customers by Store (Constrained)

📝 Instructions

Identify customers whose total spending within a store is above the average spending of that store.

Constraints:

- Only include customers with total_spend ≥ 100
- Only include stores with at least 5 such customers

📊 Expected Output

| store_id | customer_id | total_spend |
|----------|-------------| ------------|

In [40]:
ex1_data.head()

,store_id,customer_id,amount
0,1,269,0.99
1,1,274,2.99
2,1,297,0.99
3,1,344,2.99
4,2,348,0.99


My solution

In [58]:
# calculate total_spend

total_spend = ex1_data.groupby(['store_id', 'customer_id']).agg(
    total_spend = ('amount', 'sum')
).reset_index()

# filter customer by total_spend >= 100
customer = total_spend[total_spend['total_spend'] >= 100]

# filter store by customer count and calculate store average
store = customer.groupby('store_id', as_index=False).agg(
    customer_count=('customer_id', 'count'),
    avg_amount=('total_spend', 'mean')
)

store = store[store['customer_count'] >= 5]

# merge data sets
merge = customer.merge(store[['store_id', 'avg_amount']], on='store_id', how='inner')

# result comparison
result = merge[merge['total_spend'] > merge['avg_amount']]
result


,store_id,customer_id,total_spend,avg_amount
1,1,2,128.73,125.927149
2,1,3,135.74,125.927149
3,1,5,144.62,125.927149
4,1,7,151.67,125.927149
6,1,15,134.68,125.927149
...,...,...,...,...
383,2,565,126.71,125.998218
384,2,569,134.68,125.998218
387,2,575,126.71,125.998218
388,2,576,139.66,125.998218


In [56]:
result.info()

<class 'pandas.DataFrame'>
Index: 173 entries, 1 to 392
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   store_id     173 non-null    int64  
 1   customer_id  173 non-null    int64  
 2   total_spend  173 non-null    float64
 3   avg_amount   173 non-null    float64
dtypes: float64(2), int64(2)
memory usage: 6.8 KB


Solution

In [49]:
cust = ex1_data.groupby(['store_id','customer_id'], as_index=False).agg(
    total_spend=('amount','sum')
)

cust = cust[cust['total_spend'] >= 100]

valid = cust.groupby('store_id').size().reset_index(name='cnt')
valid = valid[valid['cnt'] >= 5]

cust = cust.merge(valid[['store_id']], on='store_id')

avg = cust.groupby('store_id')['total_spend'].transform('mean')

result = cust[cust['total_spend'] > avg]
result

,store_id,customer_id,total_spend
1,1,2,128.73
2,1,3,135.74
3,1,5,144.62
4,1,7,151.67
6,1,15,134.68
...,...,...,...
383,2,565,126.71
384,2,569,134.68
387,2,575,126.71
388,2,576,139.66


In [50]:
result.info()

<class 'pandas.DataFrame'>
Index: 173 entries, 1 to 392
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   store_id     173 non-null    int64  
 1   customer_id  173 non-null    int64  
 2   total_spend  173 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 5.4 KB


## Question 2

🧪 Question 2 — Declining Customer Engagement (Constrained)

📝 Instructions

Identify customer-months where rental activity decreased compared to the previous month.

Constraints:

- Only include cases where previous month rentals ≥ 5
- Only include customers with at least 3 active months

📊 Expected Output

| customer_id | month | rentals | prev_rentals |
| - | - | - | - |

My solution

In [60]:
ex2_data.head()

,customer_id,rental_date,rental_id
0,459,2022-05-24 21:54:33+00:00,2
1,408,2022-05-24 22:03:39+00:00,3
2,333,2022-05-24 22:04:41+00:00,4
3,222,2022-05-24 22:05:21+00:00,5
4,549,2022-05-24 22:08:07+00:00,6


In [61]:
ex2_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 16044 entries, 0 to 16043
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   customer_id  16044 non-null  int64              
 1   rental_date  16044 non-null  datetime64[us, UTC]
 2   rental_id    16044 non-null  int64              
dtypes: datetime64[us, UTC](1), int64(2)
memory usage: 376.2 KB


In [83]:
# extract rental month
ex2_data['month'] = ex2_data['rental_date'].dt.to_period('M')

# calculate monthly rental
rentals = ex2_data.groupby(['customer_id', 'month']).agg(
    rentals = ('rental_id', 'size')
).reset_index()

# get previous month rental
rentals = rentals.sort_values(['customer_id', 'month'])
rentals['prev_rentals'] = rentals.groupby('customer_id')['rentals'].shift(1)

# filter customers
customer = rentals.groupby('customer_id')['month'].size().reset_index(name='month_count')
customer = customer[customer['month_count'] >= 3]

# merge data sets
merge = rentals.merge(customer[['customer_id']], on='customer_id', how='inner')
result = merge[(merge['prev_rentals'] >=5) & (merge['prev_rentals'] > merge['rentals'])]
result


C:\Users\lizha\AppData\Local\Temp\ipykernel_35568\382967158.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ex2_data['month'] = ex2_data['rental_date'].dt.to_period('M')


,customer_id,month,rentals,prev_rentals
3,1,2022-08,11,12.0
7,2,2022-08,11,14.0
11,3,2022-08,7,13.0
13,4,2022-07,5,6.0
19,5,2022-08,13,16.0
...,...,...,...,...
2438,593,2022-08,8,12.0
2442,594,2022-08,3,14.0
2446,595,2022-08,8,19.0
2449,596,2022-06,2,6.0


In [84]:
result.info()

<class 'pandas.DataFrame'>
Index: 399 entries, 3 to 2459
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype    
---  ------        --------------  -----    
 0   customer_id   399 non-null    int64    
 1   month         399 non-null    period[M]
 2   rentals       399 non-null    int64    
 3   prev_rentals  399 non-null    float64  
dtypes: float64(1), int64(2), period[M](1)
memory usage: 15.6 KB


Solution

In [81]:

ex2_data['month'] = ex2_data['rental_date'].dt.to_period('M')

monthly = ex2_data.groupby(['customer_id','month']).size().reset_index(name='rentals')

valid = monthly.groupby('customer_id').size().reset_index(name='cnt')
valid = valid[valid['cnt'] >= 3]

monthly = monthly.merge(valid[['customer_id']], on='customer_id')

monthly = monthly.sort_values(['customer_id','month'])
monthly['prev_rentals'] = monthly.groupby('customer_id')['rentals'].shift(1)

result = monthly[
    (monthly['rentals'] < monthly['prev_rentals']) &
    (monthly['prev_rentals'] >= 5)
]

result

C:\Users\lizha\AppData\Local\Temp\ipykernel_35568\3220134523.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ex2_data['month'] = ex2_data['rental_date'].dt.to_period('M')


,customer_id,month,rentals,prev_rentals
3,1,2022-08,11,12.0
7,2,2022-08,11,14.0
11,3,2022-08,7,13.0
13,4,2022-07,5,6.0
19,5,2022-08,13,16.0
...,...,...,...,...
2438,593,2022-08,8,12.0
2442,594,2022-08,3,14.0
2446,595,2022-08,8,19.0
2449,596,2022-06,2,6.0


In [82]:
result.info()

<class 'pandas.DataFrame'>
Index: 399 entries, 3 to 2459
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype    
---  ------        --------------  -----    
 0   customer_id   399 non-null    int64    
 1   month         399 non-null    period[M]
 2   rentals       399 non-null    int64    
 3   prev_rentals  399 non-null    float64  
dtypes: float64(1), int64(2), period[M](1)
memory usage: 15.6 KB


## Question 3

🧪 Question 3 — Store-Level Customer Dominance (Constrained)

📝 Instructions

Identify customers whose contribution to store revenue is dominant.

Constraints:
- Only include customers with total_spend ≥ 100
- Only include stores with ≥ 5 such customers
- Only return customers ranked top 3 within store

📊 Expected Output

| store_id	|customer_id|	total_spend|	pct_contribution	|rank|
|-|-|-|-|-

My solution